<a href="https://colab.research.google.com/github/Sayak-coder/Gen-AI-codebasics-/blob/main/Equity_research_tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Retrieval

In [1]:
!pip install langchain-classic
!pip install langchain_community
!pip install langchain
!pip install sentence-transformers
!pip install faiss-cpu
!pip install langchain_groq
!pip install langchain_google_genai
!pip install langchain_openai

In [2]:
from langchain_classic.chains import RetrievalQAWithSourcesChain
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_42446/3618174644.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredURLLoader


In [3]:
loader = UnstructuredURLLoader(
    urls = [
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])
data = loader.load()

In [4]:
len(data)

2

In [5]:
r_spliter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = r_spliter.split_documents(data)

In [6]:
len(chunks)

20

In [7]:
for chunk in chunks:
  print(len(chunk.page_content))

947
943
939
998
908
892
938
989
989
979
986
203
992
940
937
994
670
988
902
861


In [8]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get('gen_ai')

In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [10]:
vectors = FAISS.from_documents(chunks, embeddings)

In [11]:
vectors

In [12]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('groq')
# os.environ["OPENAI_API_KEY"] = userdata.get('openAI')
# os.environ["GOOGLE_API_KEY"] = userdata.get('gen_ai')

In [33]:
from langchain_groq import ChatGroq

model = "openai/gpt-oss-120b"

llm = ChatGroq(
    model = model,
    groq_api_key = os.environ["GROQ_API_KEY"],
    temperature = 0.6
)

In [34]:
chain = RetrievalQAWithSourcesChain.from_llm(llm = llm, retriever=vectors.as_retriever())

In [36]:
query = "how much did Tesla (TSLA.O) rally after Morgan Stanley upgraded the electric car maker ?"

chain.invoke({"question": query}, return_only_outputs=True)

{'answer': 'Tesla’s shares jumped about **10%** after Morgan\u202fStanley upgraded the electric‑car maker to “overweight” from “equal‑weight.”  \n\n**SOURCES**  \n- “Tesla (TSLA.O) rallied 10% after Morgan Stanley upgraded the electric car maker to ‘overweight’ from ‘equal‑weight,’ …”【https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html】',
 'sources': ''}